# 02 — Invasion With Benchmark Packs and Confidence Intervals

This notebook documents the post-MVP phase:
- Live LLM strategy injection path (`llm_json`)
- Fixed held-out benchmark packs (`harsh_v1`, `harsh_v2`)
- Governance ablation with confidence intervals across repeated runs

## 1) Core Question

Can cooperation remain stable under repeated adversarial strategy injection, and which governance signal (none, monitoring, monitoring+sanctions) gives the best collapse resistance under held-out harsh regimes?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

## 2) Run Commands (from repo root)

```bash
conda activate fishery

# Optional: live LLM injection (requires OPENAI_API_KEY)
python -m experiments.run_invasion \
  --injector-mode llm_json \
  --llm-model gpt-4.1-mini \
  --benchmark-pack harsh_v1 \
  --output-prefix results/runs/invasion/invasion_live

# Governance ablation + CI over repeated runs
python -m experiments.run_governance_ablation \
  --benchmark-pack harsh_v1 \
  --n-runs 5 \
  --generations 30 \
  --seeds-per-generation 64 \
  --output-prefix results/runs/ablation/governance_ablation
```

In [ ]:
table_path = Path('../results/runs/ablation/governance_ablation_table.csv')
if not table_path.exists():
    print('Ablation table not found yet:', table_path)
else:
    table = pd.read_csv(table_path)
    display(table)

In [ ]:
if table_path.exists():
    table = pd.read_csv(table_path)
    x = table['condition']
    y = table['test_collapse_mean_mean']
    yerr_low = y - table['test_collapse_mean_ci95_low']
    yerr_high = table['test_collapse_mean_ci95_high'] - y

    plt.figure(figsize=(8, 4))
    plt.errorbar(x, y, yerr=[yerr_low, yerr_high], fmt='o', capsize=5)
    plt.ylabel('Held-out Collapse Mean (95% CI)')
    plt.xlabel('Governance Condition')
    plt.title('Governance Ablation Under Invasion Pressure')
    plt.ylim(0, 1.05)
    plt.grid(alpha=0.3)
    plt.show()

## 3) Interpretation Template

- If CI bands for `test_collapse_mean` are clearly separated, claim stronger evidence.
- If they overlap, increase `--n-runs`, `--generations`, and `--seeds-per-generation`.
- Always report benchmark pack name and regime count for reproducibility.